# Find worst performing classes by model prediction

Runs a trained model over a full split's test set, ranks glosses by per-class
f1-score, and builds a new `asl100_worst` split from the 100 worst-performing
classes.

In [ ]:
import json
from pathlib import Path

from pydantic import BaseModel

#locals
from src.configs import get_class_list
from src.preprocess import Instance
from src.run_types import AdminInfo
from src.stats import AVAIL_SETS, AVAIL_SPLITS, get_all_sets, to_preproc_format
from src.testing import get_res_path
from src.utils import plt_display_grid
from src.video_dataset import get_wlasl_info
from src.visualise2 import FrameFetcher, infer, set_thesis_style

## Setup

In [ ]:
set_thesis_style()
verbosity = 1
save_files = False
def printv(*args, level=1, **kwargs):
    if level <= verbosity:
        print(*args, **kwargs)
        
split_idx = 3 #change for different split
split_options: list[AVAIL_SPLITS] = ["asl100", "asl300", "asl1000", "asl2000"]
set_options: list[AVAIL_SETS] = ['train', 'test', 'val']
split_name: AVAIL_SPLITS = split_options[split_idx]
set_idx = 1 #change for different set
set_name: AVAIL_SETS = set_options[set_idx]
classes = get_class_list()
check_name = 'best.pth'
printv(f'Split name: {split_name}')
printv(f'Set name: {set_name}')

## Run inference and rank glosses by f1-score

In [ ]:
admin = AdminInfo.model_validate({
    "model" : "MViTv2_B_32x3",
    "dataset": "WLASL",
    "split": "asl2000",
    "save_path": "runs/asl2000/MViTv2_B_32x3/exp004/checkpoints",
    "exp_no": "004",
    "recover": False,
    "config_path": "configfiles/asl2000/MViTv2_B_32x3/exp004.toml",
    "weight_path": "runs/asl1000/MViTv2_B_32x3/exp001/checkpoints/best.pth"
})
save_path = Path(admin.save_path)

print(f"Testing on {set_name} set")
topk_res, cls_report, all_targets, all_preds = infer(admin, set_name, split_name, check_name=check_name)

In [ ]:
res_path = get_res_path(save_path)
printv(res_path)
out_dict1 = {
    'test': topk_res.model_dump(),
    'cls_report': cls_report,
}
outp1 = res_path.parent / 'topk_cls_report.json'

with open(outp1, 'w') as f:
    json.dump(out_dict1, f, indent=4)


out_dict2 = {
    'all_targets': [int(t) for t in all_targets],
    'all_preds': [int(p) for p in all_preds]
}
outp2 = res_path.parent / 'all_targets_preds.json'

with open(outp2, 'w') as f:
    json.dump(out_dict2, f)

print(f'saved to : {outp1} and {outp2}')    


In [ ]:
metric = 'f1-score'

gloss_metrics = [ 
    (key, item[metric]) for key, item in cls_report.items() if key[0].isdigit()
]

sorted_glosses = sorted(
    gloss_metrics,
    key=lambda x: x[1]
)

bottom_100_label_nums = [
    int(i) for i, _ in sorted_glosses[:100]
]

bottom_100_label_names = [classes[i] for i in bottom_100_label_nums]

printv(gloss_metrics[0])
printv(len(gloss_metrics))
printv(sorted_glosses[0])

## Preview an example clip from a worst-performing class

In [ ]:
target_length = 16
cls_idx = bottom_100_label_nums[1] # change for different class
all_sets = get_all_sets(split_name)
frame_fetcher = FrameFetcher(
    cls_idx=cls_idx,
    all_sets=all_sets,
    set_name=set_name,
    split_name=split_name,
    target_length=target_length,
)

In [ ]:
frames = frame_fetcher()
plt_display_grid(frames, num=target_length)
print(f'Example videos for class: "{classes[cls_idx]}"')
print(f'Sample: {frame_fetcher.cur_idx} / {frame_fetcher.len}')

## Build the `asl100_worst` dataset split

Remaps the 100 worst-performing glosses to labels 0-99 (keeping the original
label number for reference) and writes a new label set per split.

In [ ]:
def _pydantic_default(obj):
    if isinstance(obj, BaseModel):
        return obj.model_dump()

def map_new_labels(instances: list[Instance]) -> list[dict]:
    """Need to remap the labels to be from 0-99 instead of the original label nums
    But want to save the original label nums in the instance info as well for reference
    """
    mapped_instances = []
    for inst in instances:
        original_label_num = inst.label_num
        inst_dict = inst.model_dump()
        inst_dict['original_label_num'] = original_label_num
        inst_dict['label_num'] = bottom_100_label_nums.index(original_label_num)
        mapped_instances.append(inst_dict)
        
    return mapped_instances
        

split_name = 'asl100'
new_split_name = 'asl100_worst'
new_suffix = f'{metric}_{admin.model}_{admin.split}_{admin.exp_no}.json'
for set_name in set_options:
    bottom_100_subset = to_preproc_format(
        all_sets[set_name],
        criterion=lambda x: x['gloss'] in bottom_100_label_names
    )
    bottom_100_mapped = map_new_labels(bottom_100_subset)
    
    
    
    original_set_path_info = get_wlasl_info(split_name, set_name)
    new_dir = original_set_path_info['labels'].parent / new_split_name 
    # new_dir = Path('preprocessed/labels/asl_100_worst/')
    new_path = new_dir / f'{set_name}_{new_suffix}'
    
    new_dir.mkdir(exist_ok=True, parents=True)
    with open(new_path, 'w') as f:
        json.dump(bottom_100_mapped, f, indent=2, default=_pydantic_default)
    printv(f'Saved to: {new_path}')